In [1]:
%load_ext autoreload
%autoreload 2

In [73]:
import jax.numpy as jnp
import jax
import jax.random as random
from flax import linen as nn
import tensorflow as tf
import optax
from tqdm import tqdm
from utils import DataLoader
from utils import MLP

In [74]:
key = random.PRNGKey(0) # chiave per random 

f_to_learn = lambda mu, k, l, x: jnp.sin(2*mu*jnp.pi*x) + k + jnp.exp(l*x)
N = 10000

key, subkey = random.split(key) # ogni volta, prima di usare la chiave, la devi dividere
x = random.uniform(subkey, (N,), minval=-10, maxval=10)
key, subkey = random.split(key)
mu = random.uniform(subkey, (N,), minval=-2, maxval=2)
key, subkey = random.split(key)
k = random.uniform(subkey, (N,), minval=-5, maxval=5)
key, subkey = random.split(key)
l = random.uniform(subkey, (N,), minval=-1, maxval=1)

y = f_to_learn(mu, k, l, x) # così generiamo artificialmente un dataset di N punti

In [75]:
X = jnp.stack([mu, k, l, x], axis=1)

In [76]:
X = jnp.stack([mu, k, l, x], axis=1)


split_idx = int(N * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

train_dataloader = DataLoader(X_train, y_train, batch_size=32, shuffle=True)
test_dataloader = DataLoader(X_test, y_test, batch_size=32, shuffle=False)


In [77]:
# Example of iterating through the DataLoader
for data, label in train_dataloader:
    print(data.shape, label.shape)
    break

(1, 32, 4) (1, 32)


In [78]:
targetnetwork = MLP(output_dim=1, hidden_dim=8, num_hidden_layers=1)

In [79]:
x = jnp.ones((1,1)) #Gli input sono SEMPRE (SEMPRE) nel formato (bathc_size, input_dim1, input_dim2, ..., input_dimN)
# In questo caso, batch_size=1, input_dim=1
key = jax.random.PRNGKey(0) # Bisogna sempre passare una key per inizializzare i pesi random
print(targetnetwork.tabulate(key, x)) # Visualizza la struttura del modello, con i pesi inizializzati


                               MLP Summary                               
┏━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ path    ┃ module ┃ inputs       ┃ outputs      ┃ params               ┃
┡━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│         │ MLP    │ float32[1,1] │ float32[1,1] │                      │
├─────────┼────────┼──────────────┼──────────────┼──────────────────────┤
│ Dense_0 │ Dense  │ float32[1,1] │ float32[1,8] │ bias: float32[8]     │
│         │        │              │              │ kernel: float32[1,8] │
│         │        │              │              │                      │
│         │        │              │              │ 16 (64 B)            │
├─────────┼────────┼──────────────┼──────────────┼──────────────────────┤
│ Dense_1 │ Dense  │ float32[1,8] │ float32[1,1] │ bias: float32[1]     │
│         │        │              │              │ kernel: float32[8,1] │
│         │        │              │  

In [80]:
hypernetwork = MLP(output_dim = 25, hidden_dim=8, num_hidden_layers=2) # 25 come i parametri del target network

## First try: only training a single network

In [109]:
key = jax.random.PRNGKey(0)
key, subkey = random.split(key)
toy_data = random.uniform(key, (1000, 1), minval=-3, maxval=3)
toy_label = f_to_learn(0.5, 1.0, 0.1, toy_data)
train_dataloader = DataLoader(toy_data, toy_label, batch_size=32, shuffle=True)

In [114]:
model = MLP(output_dim=1, hidden_dim=8, num_hidden_layers=1)

In [ ]:
def mse_loss(preds, targets):
    return jnp.mean((preds - targets) ** 2)

optimizer = optax.adam(learning_rate=1e-3)
params = model.init(jax.random.PRNGKey(0), jnp.zeros((1, 1)))
opt_state = optimizer.init(params)
epochs = 100

In [113]:
params

{'params': {'Dense_0': {'kernel': Array([[-1.6517996 ,  0.7545515 , -1.3777502 , -0.852023  ,  0.9193905 ,
            1.2836219 ,  1.0440549 , -0.04649625]], dtype=float32),
   'bias': Array([0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)},
  'Dense_1': {'kernel': Array([[ 0.00117643],
          [ 0.11322609],
          [ 0.06392855],
          [-0.2971922 ],
          [ 0.22761364],
          [ 0.3089246 ],
          [-0.3945709 ],
          [-0.25477833]], dtype=float32),
   'bias': Array([0.], dtype=float32)}}}

In [117]:
next(train_dataloader)

(Array([[ 2.7746286 ,  0.79780984, -1.2244799 , -1.4952536 ,  0.4531889 ,
         -1.3779187 ,  0.67894363, -1.6148651 , -2.7891533 ,  2.22107   ,
         -2.746183  ,  1.0254178 ,  1.6205964 ,  0.72274375, -2.9485552 ,
         -2.2777798 ,  2.2315235 , -2.8311217 , -2.0763638 ,  1.0911899 ,
          2.638479  ,  1.5838716 ,  0.3870356 ,  0.8875301 ,  0.53722215,
         -1.7657776 , -0.8722322 , -0.40641975, -2.4540932 , -1.8909566 ,
         -1.8064542 ,  0.8218374 ]], dtype=float32),
 Array([[2.9701118, 2.6763878, 2.5329566, 2.8610055, 3.0355673, 2.7986295,
         2.9163537, 2.7864718, 1.1415973, 2.8887157, 1.0443269, 2.028216 ,
         1.2468452, 2.8399405, 1.5837235, 1.0302522, 2.9149055, 1.2474308,
         1.5748929, 1.8327162, 3.2087815, 1.2061331, 2.9771476, 2.438839 ,
         3.0483623, 2.509337 , 1.5257703, 1.0030777, 0.7927659, 2.1636157,
         2.4059927, 2.6165993]], dtype=float32))

In [112]:
def train_step(model, params, opt_state, loss, optimizer, x, y, key=None):
    loss_fn = lambda p, x, y: loss(model.apply(p, x), y)
    loss, grad = jax.value_and_grad(loss_fn)(params, x, y)
    
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)

    return params, opt_state, loss

train_step = jax.jit(train_step, static_argnames=('model', 'loss', 'optimizer'))

for _ in tqdm(range(epochs)):
    epoch_loss = 0.0
    for data, label in train_dataloader:
        params, opt_state, loss = train_step(model = model, params = params, opt_state = opt_state, loss = mse_loss, optimizer = optimizer, x = data, y = label)
        epoch_loss += loss
    print(f"Epoch Loss: {epoch_loss / len(train_dataloader)}")



  0%|          | 0/100 [00:00<?, ?it/s]


ScopeParamShapeError: Initializer expected to generate shape (1, 8) but got shape (32, 8) instead for parameter "kernel" in "/Dense_0". (https://flax.readthedocs.io/en/latest/api_reference/flax.errors.html#flax.errors.ScopeParamShapeError)

In [140]:
labels = random.uniform(key, (10, 1), minval=-3, maxval=3)
labels = labels.squeeze()
labels

a = labels[3:6]
a

Array([-2.2756462, -1.8491192,  1.3320901], dtype=float32)

In [126]:
jnp.expand_dims(a, axis=-1)

Array([[-2.2756462],
       [-1.8491192],
       [ 1.3320901]], dtype=float32)

In [138]:
labels = random.uniform(key, (10, 3), minval=-3, maxval=3)
labels = labels.squeeze()
labels

a = labels[1:6, :]
a

Array([[-2.2756462 , -1.8491192 ,  1.3320901 ],
       [ 1.5926735 , -2.0847573 ,  2.7102377 ],
       [-2.8241372 , -2.4077368 ,  0.31885958],
       [-2.2533174 ,  0.5673723 ,  2.7569447 ],
       [ 1.159363  ,  1.3445756 , -1.0910139 ]], dtype=float32)

In [139]:
a.shape

(5, 3)

In [130]:
jnp.expand_dims(a, axis=-1)

Array([[[-2.2756462 ],
        [-1.8491192 ],
        [ 1.3320901 ]],

       [[ 1.5926735 ],
        [-2.0847573 ],
        [ 2.7102377 ]],

       [[-2.8241372 ],
        [-2.4077368 ],
        [ 0.31885958]],

       [[-2.2533174 ],
        [ 0.5673723 ],
        [ 2.7569447 ]],

       [[ 1.159363  ],
        [ 1.3445756 ],
        [-1.0910139 ]]], dtype=float32)

https://huggingface.co/blog/afmck/flax-tutorial
https://wandb.ai/jax-series/simple-training-loop/reports/Writing-a-Training-Loop-in-JAX-and-Flax--VmlldzoyMzA4ODEy